In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import string, math, random
import numpy as np
import torch
from scipy.sparse import csr_matrix, diags
from collections import defaultdict, Counter
from sklearn.metrics import f1_score
from sklearn.feature_selection import chi2
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print('Imports done.')

Device : cuda
GPU    : NVIDIA GeForce RTX 5080 Laptop GPU
Imports done.


In [2]:
# ── Cell 2: Load Data ─────────────────────────────────────────────────────────
def load_train(path):
    labels, texts = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split(' ', 1)
                labels.append(int(parts[0]))
                texts.append(parts[1] if len(parts) > 1 else '')
    return labels, texts

def load_test(path):
    texts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                texts.append(line)
    return texts

train_labels, train_texts = load_train('train.dat')
test_texts = load_test('test.dat')
print(f'Training samples : {len(train_labels)}')
print(f'Test samples     : {len(test_texts)}')
print(f'Classes          : {sorted(set(train_labels))}')

Training samples : 102080
Test samples     : 25520
Classes          : [1, 2, 3, 4]


In [3]:
# ── Cell 3: Pre-processing (unigrams + bigrams + trigrams) ────────────────────
STOP_WORDS  = set(stopwords.words('english'))
stemmer     = PorterStemmer()
PUNCT_TABLE = str.maketrans('', '', string.punctuation)

def preprocess(text):
    text     = text.lower().translate(PUNCT_TABLE)
    unigrams = [stemmer.stem(t) for t in text.split()
                if t not in STOP_WORDS and len(t) > 1]
    bigrams  = [f'{unigrams[i]}_{unigrams[i+1]}'
                for i in range(len(unigrams) - 1)]
    trigrams = [f'{unigrams[i]}_{unigrams[i+1]}_{unigrams[i+2]}'
                for i in range(len(unigrams) - 2)]
    return unigrams + bigrams + trigrams

print(preprocess('oil prices rise in the stock market today'))

['oil', 'price', 'rise', 'stock', 'market', 'today', 'oil_price', 'price_rise', 'rise_stock', 'stock_market', 'market_today', 'oil_price_rise', 'price_rise_stock', 'rise_stock_market', 'stock_market_today']


In [4]:
# ── Cell 4: Apply Pre-processing ─────────────────────────────────────────────
print('Preprocessing training data...')
train_tokens = [preprocess(t) for t in train_texts]
print('Preprocessing test data...')
test_tokens  = [preprocess(t) for t in test_texts]
print(f'Done. Sample: {train_tokens[0][:8]}')

Preprocessing training data...
Preprocessing test data...
Done. Sample: ['town', 'celebr', 'hope', 'amir', 'strike', 'gold', 'amir', 'khan']


In [5]:
# ── Cell 5: Helper Functions ──────────────────────────────────────────────────
# All reusable functions defined here so every later cell can use them.

def renormalize(mat):
    """L2-normalize each row of a sparse matrix."""
    norms = np.sqrt(mat.multiply(mat).sum(axis=1)).A1
    norms[norms == 0] = 1
    return diags(1.0 / norms) @ mat

def build_vocab_idf(token_lists, min_df=2, max_vocab=150000, mode='bm25'):
    """Build vocabulary and IDF array from token lists."""
    N = len(token_lists)
    doc_freq = defaultdict(int)
    for toks in token_lists:
        for t in set(toks):
            doc_freq[t] += 1
    terms = sorted([(t, df) for t, df in doc_freq.items() if df >= min_df],
                   key=lambda x: -x[1])[:max_vocab]
    vocab = {t: i for i, (t, _) in enumerate(terms)}
    idf   = np.zeros(len(vocab), dtype=np.float32)
    for t, idx in vocab.items():
        df = doc_freq[t]
        if mode == 'bm25':
            idf[idx] = math.log((N - df + 0.5) / (df + 0.5) + 1)
        else:
            idf[idx] = math.log((N + 1) / (df + 1)) + 1
    return vocab, idf, doc_freq

def build_bm25_matrix(token_lists, vocab, idf, k1, b, avgdl):
    """Build BM25-weighted L2-normalized sparse matrix."""
    rows, cols, vals = [], [], []
    for doc_idx, tokens in enumerate(token_lists):
        if not tokens: continue
        tf_counts = Counter(tokens)
        doc_len   = len(tokens)
        for term, tf in tf_counts.items():
            if term in vocab:
                col   = vocab[term]
                score = idf[col] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len / avgdl))
                rows.append(doc_idx); cols.append(col); vals.append(score)
    mat = csr_matrix((vals, (rows, cols)), shape=(len(token_lists), len(vocab)), dtype=np.float32)
    return renormalize(mat)

def build_tfidf_matrix(token_lists, vocab, idf):
    """Build sublinear TF-IDF L2-normalized sparse matrix."""
    rows, cols, vals = [], [], []
    for doc_idx, tokens in enumerate(token_lists):
        if not tokens: continue
        tf_counts = Counter(tokens)
        total     = len(tokens)
        for term, count in tf_counts.items():
            if term in vocab:
                col = vocab[term]
                rows.append(doc_idx); cols.append(col)
                vals.append((1 + math.log(count)) / total * idf[col])
    mat = csr_matrix((vals, (rows, cols)), shape=(len(token_lists), len(vocab)), dtype=np.float32)
    return renormalize(mat)

def apply_chi2(train_mat, test_mat, labels, k):
    """Select top-k features by chi-squared score, re-normalize."""
    scores, _ = chi2(train_mat, labels)
    top = np.argsort(scores)[-k:]; top.sort()
    return renormalize(train_mat[:, top].tocsr()), renormalize(test_mat[:, top].tocsr())

def scipy_sparse_to_torch(mat, device):
    mat = mat.tocsr().astype(np.float32)
    mat.sort_indices(); mat.sum_duplicates()
    crow   = torch.tensor(mat.indptr,  dtype=torch.int64)
    col    = torch.tensor(mat.indices, dtype=torch.int64)
    values = torch.tensor(mat.data,    dtype=torch.float32)
    return torch.sparse_csr_tensor(crow, col, values, size=mat.shape).to(device)

def knn_predict(test_mat, train_mat, train_labels, k, batch_size=200):
    """k-NN with distance-weighted voting — GPU accelerated via PyTorch."""
    train_gpu        = scipy_sparse_to_torch(train_mat, DEVICE)
    train_labels_arr = np.array(train_labels)
    predictions      = []
    for start in range(0, test_mat.shape[0], batch_size):
        end         = min(start + batch_size, test_mat.shape[0])
        batch_dense = torch.tensor(test_mat[start:end].toarray(),
                                   dtype=torch.float32, device=DEVICE)
        sims              = torch.sparse.mm(train_gpu, batch_dense.T).T
        top_vals, top_idx = torch.topk(sims, k, dim=1)
        for sim_scores, idx_row in zip(top_vals.cpu().numpy(), top_idx.cpu().numpy()):
            top_k_labels = train_labels_arr[idx_row]
            weights      = defaultdict(float)
            for label, w in zip(top_k_labels, sim_scores):
                weights[label] += float(w)
            predictions.append(max(weights, key=weights.get))
        if (start // batch_size) % 25 == 0:
            print(f'  {end}/{test_mat.shape[0]} done...')
    return predictions

def best_k_search(val_mat, fit_mat, fit_labels, val_labels, k_list):
    best_k, best_f1 = 1, 0.0
    for k in k_list:
        preds = knn_predict(val_mat, fit_mat, fit_labels, k=k)
        f1    = f1_score(val_labels, preds, average='macro')
        print(f'  k={k}  F1={f1:.4f}')
        if f1 > best_f1:
            best_f1, best_k = f1, k
    return best_k, best_f1

print('All helper functions defined.')

All helper functions defined.


In [6]:
# ── Cell 6: BM25 Parameter Search (quick subset) ─────────────────────────────
# Find best k1 and b on 12k samples before building full matrices.

MIN_DF  = 2
BM25_K1 = 1.5
BM25_B  = 0.75
N       = len(train_tokens)
avgdl   = np.mean([len(t) for t in train_tokens])

# Shared vocab — larger for Model 1 (trigrams need more room)
vocab, idf, doc_freq = build_vocab_idf(train_tokens, min_df=MIN_DF,
                                        max_vocab=200000, mode='bm25')
print(f'Vocab size: {len(vocab)}  avgdl: {avgdl:.1f}')

random.seed(42)
q_idx = list(range(12000)); random.shuffle(q_idx)
q_fit_idx, q_val_idx = q_idx[:10000], q_idx[10000:]
q_fit_labels = [train_labels[i] for i in q_fit_idx]
q_val_labels = [train_labels[i] for i in q_val_idx]

best_params, best_qf1 = (BM25_K1, BM25_B), 0.0
for k1 in [1.2, 1.5, 2.0]:
    for b in [0.5, 0.75, 1.0]:
        qtr = build_bm25_matrix([train_tokens[i] for i in q_fit_idx], vocab, idf, k1, b, avgdl)
        qva = build_bm25_matrix([train_tokens[i] for i in q_val_idx], vocab, idf, k1, b, avgdl)
        preds = knn_predict(qva, qtr, q_fit_labels, k=11, batch_size=200)
        f1    = f1_score(q_val_labels, preds, average='macro')
        print(f'  k1={k1}  b={b}  F1={f1:.4f}')
        if f1 > best_qf1:
            best_qf1, best_params = f1, (k1, b)

BM25_K1, BM25_B = best_params
print(f'\nBest: k1={BM25_K1}  b={BM25_B}')

Vocab size: 200000  avgdl: 74.0


C:\Users\zp123_2zkvvkz\AppData\Local\Temp\ipykernel_48792\4275489207.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  return torch.sparse_csr_tensor(crow, col, values, size=mat.shape).to(device)
C:\Users\zp123_2zkvvkz\AppData\Local\Temp\ipykernel_48792\4275489207.py:71: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_cs

  200/2000 done...
  k1=1.2  b=0.5  F1=0.8672
  200/2000 done...
  k1=1.2  b=0.75  F1=0.8656
  200/2000 done...
  k1=1.2  b=1.0  F1=0.8652
  200/2000 done...
  k1=1.5  b=0.5  F1=0.8655
  200/2000 done...
  k1=1.5  b=0.75  F1=0.8661
  200/2000 done...
  k1=1.5  b=1.0  F1=0.8657
  200/2000 done...
  k1=2.0  b=0.5  F1=0.8661
  200/2000 done...
  k1=2.0  b=0.75  F1=0.8671
  200/2000 done...
  k1=2.0  b=1.0  F1=0.8671

Best: k1=1.2  b=0.5


In [7]:
# ── Cell 7: Build Model 1 Matrix (BM25 + uni+bi+trigrams + chi2) ──────────────
print('Building Model 1 (BM25 + trigrams)...')
train_m1_full = build_bm25_matrix(train_tokens, vocab, idf, BM25_K1, BM25_B, avgdl)
test_m1_full  = build_bm25_matrix(test_tokens,  vocab, idf, BM25_K1, BM25_B, avgdl)
train_m1, test_m1 = apply_chi2(train_m1_full, test_m1_full, train_labels, k=150000)
print(f'M1 shape: {train_m1.shape}')

Building Model 1 (BM25 + trigrams)...
M1 shape: (102080, 150000)


In [8]:
# ── Cell 8: Build Model 2 Matrix (sublinear TF-IDF + uni+bigrams + chi2) ──────
print('Building Model 2 (TF-IDF + bigrams)...')
train_tokens_m2 = [[t for t in toks if t.count('_') <= 1] for toks in train_tokens]
test_tokens_m2  = [[t for t in toks if t.count('_') <= 1] for toks in test_tokens]
vocab_m2, idf_m2, _ = build_vocab_idf(train_tokens_m2, min_df=MIN_DF,
                                        max_vocab=150000, mode='tfidf')
train_m2_full = build_tfidf_matrix(train_tokens_m2, vocab_m2, idf_m2)
test_m2_full  = build_tfidf_matrix(test_tokens_m2,  vocab_m2, idf_m2)
train_m2, test_m2 = apply_chi2(train_m2_full, test_m2_full, train_labels, k=120000)
print(f'M2 shape: {train_m2.shape}')

Building Model 2 (TF-IDF + bigrams)...
M2 shape: (102080, 120000)


In [9]:
# ── Cell 9: Build Model 3 Matrix (BM25 + unigrams only + chi2) ───────────────
print('Building Model 3 (BM25 + unigrams)...')
train_tokens_m3 = [[t for t in toks if '_' not in t] for toks in train_tokens]
test_tokens_m3  = [[t for t in toks if '_' not in t] for toks in test_tokens]
vocab_m3, idf_m3, _ = build_vocab_idf(train_tokens_m3, min_df=MIN_DF,
                                        max_vocab=50000, mode='bm25')
avgdl_m3      = np.mean([len(t) for t in train_tokens_m3])
train_m3_full = build_bm25_matrix(train_tokens_m3, vocab_m3, idf_m3, BM25_K1, BM25_B, avgdl_m3)
test_m3_full  = build_bm25_matrix(test_tokens_m3,  vocab_m3, idf_m3, BM25_K1, BM25_B, avgdl_m3)
train_m3, test_m3 = apply_chi2(train_m3_full, test_m3_full, train_labels,
                                k=min(50000, train_m3_full.shape[1]))
print(f'M3 shape: {train_m3.shape}')

Building Model 3 (BM25 + unigrams)...
M3 shape: (102080, 35566)


In [10]:
# ── Cell 9b: Build Model 4 Matrix (BM25 + uni+bigrams + chi2) ────────────────
# Adds ensemble diversity: BM25 scoring on uni+bigrams (no trigrams, no TF-IDF)
print('Building Model 4 (BM25 + bigrams)...')
train_tokens_m4 = [[t for t in toks if t.count('_') <= 1] for toks in train_tokens]
test_tokens_m4  = [[t for t in toks if t.count('_') <= 1] for toks in test_tokens]
vocab_m4, idf_m4, _ = build_vocab_idf(train_tokens_m4, min_df=MIN_DF,
                                        max_vocab=150000, mode='bm25')
avgdl_m4      = np.mean([len(t) for t in train_tokens_m4])
train_m4_full = build_bm25_matrix(train_tokens_m4, vocab_m4, idf_m4, BM25_K1, BM25_B, avgdl_m4)
test_m4_full  = build_bm25_matrix(test_tokens_m4,  vocab_m4, idf_m4, BM25_K1, BM25_B, avgdl_m4)
train_m4, test_m4 = apply_chi2(train_m4_full, test_m4_full, train_labels,
                                k=min(120000, train_m4_full.shape[1]))
print(f'M4 shape: {train_m4.shape}')

Building Model 4 (BM25 + bigrams)...
M4 shape: (102080, 120000)


In [11]:
# ── Cell 10: Validate All 4 Models + Ensemble ─────────────────────────────────
random.seed(42)
indices = list(range(len(train_labels))); random.shuffle(indices)
split   = int(0.8 * len(indices))
fit_idx, val_idx = indices[:split], indices[split:]
fit_labels = [train_labels[i] for i in fit_idx]
val_labels = [train_labels[i] for i in val_idx]

K_LIST = [11] + list(range(21, 42))  # k=11 + every k from 21 to 41

print('--- Model 1 ---')
best_k_m1, best_f1_m1 = best_k_search(
    train_m1[val_idx], train_m1[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M1 best: k={best_k_m1}  F1={best_f1_m1:.4f}\n')

print('--- Model 2 ---')
best_k_m2, best_f1_m2 = best_k_search(
    train_m2[val_idx], train_m2[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M2 best: k={best_k_m2}  F1={best_f1_m2:.4f}\n')

print('--- Model 3 ---')
best_k_m3, best_f1_m3 = best_k_search(
    train_m3[val_idx], train_m3[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M3 best: k={best_k_m3}  F1={best_f1_m3:.4f}\n')

print('--- Model 4 ---')
best_k_m4, best_f1_m4 = best_k_search(
    train_m4[val_idx], train_m4[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M4 best: k={best_k_m4}  F1={best_f1_m4:.4f}\n')

# Ensemble validation — weighted by each model's best F1
print('--- Ensemble ---')
vp1 = knn_predict(train_m1[val_idx], train_m1[fit_idx], fit_labels, k=best_k_m1)
vp2 = knn_predict(train_m2[val_idx], train_m2[fit_idx], fit_labels, k=best_k_m2)
vp3 = knn_predict(train_m3[val_idx], train_m3[fit_idx], fit_labels, k=best_k_m3)
vp4 = knn_predict(train_m4[val_idx], train_m4[fit_idx], fit_labels, k=best_k_m4)

w1, w2, w3, w4 = best_f1_m1, best_f1_m2, best_f1_m3, best_f1_m4
ens_val = []
for p1, p2, p3, p4 in zip(vp1, vp2, vp3, vp4):
    wts = defaultdict(float)
    wts[p1] += w1; wts[p2] += w2; wts[p3] += w3; wts[p4] += w4
    ens_val.append(max(wts, key=wts.get))

ens_f1 = f1_score(val_labels, ens_val, average='macro')
print(f'Ensemble F1 = {ens_f1:.4f}  (M1={best_f1_m1:.4f}  M2={best_f1_m2:.4f}  M3={best_f1_m3:.4f}  M4={best_f1_m4:.4f})')

--- Model 1 ---
  200/20416 done...
  5200/20416 done...
  10200/20416 done...
  15200/20416 done...
  20200/20416 done...
  k=11  F1=0.9182
  200/20416 done...
  5200/20416 done...
  10200/20416 done...
  15200/20416 done...
  20200/20416 done...
  k=21  F1=0.9182
  200/20416 done...
  5200/20416 done...
  10200/20416 done...
  15200/20416 done...
  20200/20416 done...
  k=22  F1=0.9180
  200/20416 done...
  5200/20416 done...
  10200/20416 done...
  15200/20416 done...
  20200/20416 done...
  k=23  F1=0.9183
  200/20416 done...
  5200/20416 done...
  10200/20416 done...
  15200/20416 done...
  20200/20416 done...
  k=24  F1=0.9186
  200/20416 done...
  5200/20416 done...
  10200/20416 done...
  15200/20416 done...
  20200/20416 done...
  k=25  F1=0.9178
  200/20416 done...
  5200/20416 done...
  10200/20416 done...
  15200/20416 done...
  20200/20416 done...
  k=26  F1=0.9175
  200/20416 done...
  5200/20416 done...
  10200/20416 done...
  15200/20416 done...
  20200/20416 done...
  

In [12]:
# ── Cell 11: Final Predictions (4-model ensemble on full training data) ────────
print('M1 on full test...')
pred_m1 = knn_predict(test_m1, train_m1, train_labels, k=best_k_m1)
print('M2 on full test...')
pred_m2 = knn_predict(test_m2, train_m2, train_labels, k=best_k_m2)
print('M3 on full test...')
pred_m3 = knn_predict(test_m3, train_m3, train_labels, k=best_k_m3)
print('M4 on full test...')
pred_m4 = knn_predict(test_m4, train_m4, train_labels, k=best_k_m4)

test_predictions = []
for p1, p2, p3, p4 in zip(pred_m1, pred_m2, pred_m3, pred_m4):
    wts = defaultdict(float)
    wts[p1] += w1; wts[p2] += w2; wts[p3] += w3; wts[p4] += w4
    test_predictions.append(int(max(wts, key=wts.get)))

print(f'Total: {len(test_predictions)}  Distribution: {Counter(test_predictions)}')

M1 on full test...
  200/25520 done...
  5200/25520 done...
  10200/25520 done...
  15200/25520 done...
  20200/25520 done...
  25200/25520 done...
M2 on full test...
  200/25520 done...
  5200/25520 done...
  10200/25520 done...
  15200/25520 done...
  20200/25520 done...
  25200/25520 done...
M3 on full test...
  200/25520 done...
  5200/25520 done...
  10200/25520 done...
  15200/25520 done...
  20200/25520 done...
  25200/25520 done...
M4 on full test...
  200/25520 done...
  5200/25520 done...
  10200/25520 done...
  15200/25520 done...
  20200/25520 done...
  25200/25520 done...
Total: 25520  Distribution: Counter({2: 6603, 4: 6473, 3: 6272, 1: 6172})


In [13]:
# ── Cell 12: Save Output ──────────────────────────────────────────────────────
with open('predictions.dat', 'w') as f:
    for pred in test_predictions:
        f.write(f'{pred}\n')

with open('format.dat') as f:
    n_fmt = sum(1 for _ in f)

print(f'Saved predictions.dat — {len(test_predictions)} lines')
print('Match!' if len(test_predictions) == n_fmt else 'WARNING: line count mismatch!')

Saved predictions.dat — 25520 lines
Match!
